# Stage 4: Numerical Analysis (Regression Models)

For each regressor in `REGRESSION_MODELS` × each seed in `CONFIG["random_seeds"]`, fits and evaluates **two variants** of the same model on the same chronological train/test split:

- **Baseline**: numerical features only (returns, volatility, drawdown, fundamental ratios, log assets).
- **+NLP**: baseline features plus rolling K-day mean class probabilities from the best Stage 3 NLP model (`artifacts/nlp_probs.parquet`).

## Why this comparison matters
The baseline-vs-+NLP delta on the same model, same seed, same split is the project's primary signal-uplift question: **does sentiment add information about future realized volatility, after controlling for price-based features?** Reporting per-model, per-seed deltas separates genuine NLP uplift from seed-level noise, and Stage 6 averages these across seeds to give a single headline R² improvement per regressor.

## What this stage produces
`./results/regression_results.csv`: one row per (model, variant, seed) with R², RMSE, MAE, MSE, train/inference times, sample count, and ISO 8601 UTC start/end timestamps for both phases (used by the external energy join).

## What this stage consumes
- `./artifacts/num_df.parquet` (Stage 1): numerical features and risk_score target.
- `./artifacts/text_df.parquet` (Stage 1): only used to compute the shared cutoff.
- `./artifacts/nlp_probs.parquet` (Stage 3): per-headline class probabilities.

## Caveats worth disclosing in the writeup
- **Numerical test set is small (~46 rows).** This is a consequence of the FNSPID coverage limitation (KO headlines only span 2020-04 to 2023-12), which compresses the overlapping date range used for splits.
- **For dates with no headline coverage**, `attach_nlp_prob_features` falls back to neutral 1/3 priors, so the +NLP variant's NLP features are effectively constant for most pre-2020 rows.


In [1]:
# Shared helpers/config plus model registries.
from common import *

# Regression metrics for evaluation.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# inspect: checks whether a model factory expects a seed.
import inspect
# datetime: wall-clock ISO timestamps for external energy join.
from datetime import datetime, timezone

# Load artifacts from previous stages.
text_df = load_text_df()
num_df = load_num_df()
# Per-headline probabilities from Stage 3.
nlp_probs_df = load_nlp_probs_df()

# Date granularity for consistent splitting/joining.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Shared chronological split boundary so text and numeric eval align.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)

# Build the +NLP numerical table by attaching rolling K-day mean sentiment
# probabilities to each numerical row. For dates with no headlines in the
# lookback window (most pre-2020 rows, given FNSPID's 2020-04+ KO coverage),
# this helper falls back to neutral 1/3 priors for each class. That means
# the +NLP variant for those rows has NLP features that are effectively
# constant: any uplift it shows comes entirely from the post-2020 segment.
num_aug_df = attach_nlp_prob_features(
    num_df=num_df,
    nlp_probs_df=nlp_probs_df,
    lookback_days=CONFIG["regression"]["nlp_lookback_days"],
    date_col="date",
)

# Per-run record collector saved at end of stage.
results = ResultsCollector()

[common] device=cuda  artifacts=/home/hrilab/energy-analysis-pipeline/artifacts  results=/home/hrilab/energy-analysis-pipeline/results


## 4.1 Training routine

In [2]:
def train_regression_model(model_name, model_factory, X_tr, X_te, y_tr, y_te, seed):
    """Train one regression model for one seed and return metrics + fitted model."""

    # The REGRESSION_MODELS registry mixes factories of two shapes: sklearn
    # linear models that don't need a seed (LinearRegression, Ridge, Lasso,
    # ElasticNet are deterministic given the data) and gradient-boosted models
    # that do (XGBoost / LightGBM / CatBoost use random_state for column
    # subsampling and tie-breaking). Rather than maintain two parallel call
    # paths, the factory's signature is introspected: if it accepts any
    # arguments, pass `seed`; otherwise call it argument-less. This keeps the
    # registry definition trivial in common.py.
    sig = inspect.signature(model_factory)
    model = model_factory(seed) if sig.parameters else model_factory()

    # Fit time + wall-clock anchors for external energy join.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Prediction time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    y_pred = model.predict(X_te)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # mse is used directly and also converted to rmse.
    mse = mean_squared_error(y_te, y_pred)
    return {
        "model": model_name,
        "seed": seed,
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_te, y_pred),
        "r2": r2_score(y_te, y_pred),
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(y_te),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }, model

## 4.2 Run every (model, seed) × (baseline, +NLP) combination

In [3]:
# trained_reg_models: built but never persisted and never read downstream.
# Kept for optional in-notebook diagnostics (e.g., feature-importance plots
# on a specific tree-based regressor). Safe to remove if unused.
trained_reg_models = {}

# Baseline numerical split (no NLP features).
(
    X_tr_base,
    X_te_base,
    y_tr,
    y_te,
    _,
    base_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Augmented numerical split (includes rolling NLP probabilities).
(
    X_tr_aug,
    X_te_aug,
    y_tr_aug,
    y_te_aug,
    _,
    aug_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_aug_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Defensive check. If baseline and augmented splits started disagreeing on
# row ordering or target values (e.g., a future refactor introduces a
# different sort key inside one of the split helpers), the per-row baseline-
# vs-+NLP comparison would become meaningless: the same (model, seed) would
# be predicting on different rows in each variant. This guard catches that
# class of bug at runtime rather than letting it silently corrupt the uplift
# table.
if not np.array_equal(y_tr, y_tr_aug) or not np.array_equal(y_te, y_te_aug):
    raise ValueError("Baseline and +NLP splits do not align on targets.")

print(f"Baseline feature count: {len(base_feature_cols)}")
print(f"Augmented feature count: {len(aug_feature_cols)}")

# Train every model for every seed for baseline and +NLP variants.
for model_name, factory in REGRESSION_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        baseline_record, baseline_model = train_regression_model(
            model_name=model_name,
            model_factory=factory,
            X_tr=X_tr_base,
            X_te=X_te_base,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(baseline_record)
        trained_reg_models[(model_name, seed, "baseline")] = baseline_model

        augmented_record, augmented_model = train_regression_model(
            model_name=f"{model_name} +NLP",
            model_factory=factory,
            X_tr=X_tr_aug,
            X_te=X_te_aug,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(augmented_record)
        trained_reg_models[(model_name, seed, "+NLP")] = augmented_model

        # Side-by-side R² so uplift is easy to inspect while running.
        print(
            f"{model_name} (seed {seed})  "
            f"baseline R²={baseline_record['r2']:.4f}, "
            f"+NLP R²={augmented_record['r2']:.4f}"
        )

Baseline feature count: 11
Augmented feature count: 14
Linear Regression (seed 0)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 1)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 2)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 3)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 4)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 5)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 6)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 7)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 8)  baseline R²=-0.0131, +NLP R²=-0.0083
Linear Regression (seed 9)  baseline R²=-0.0131, +NLP R²=-0.0083
Ridge (seed 0)  baseline R²=-0.0130, +NLP R²=-0.0081
Ridge (seed 1)  baseline R²=-0.0130, +NLP R²=-0.0081
Ridge (seed 2)  baseline R²=-0.0130, +NLP R²=-0.0081
Ridge (seed 3)  baseline R²=-0.0130, +NLP R²=-0.0081
Ridge (seed 4)  baseline R²=-0.0130, +NLP R²=-0.0081
Ridge (seed 5)  baseline R²=-0

Elastic Net (seed 9)  baseline R²=-0.0003, +NLP R²=-0.0003


XGBoost (seed 0)  baseline R²=-0.1331, +NLP R²=-0.3845
XGBoost (seed 1)  baseline R²=-0.1331, +NLP R²=-0.3845


XGBoost (seed 2)  baseline R²=-0.1331, +NLP R²=-0.3845
XGBoost (seed 3)  baseline R²=-0.1331, +NLP R²=-0.3845
XGBoost (seed 4)  baseline R²=-0.1331, +NLP R²=-0.3845


XGBoost (seed 5)  baseline R²=-0.1331, +NLP R²=-0.3845
XGBoost (seed 6)  baseline R²=-0.1331, +NLP R²=-0.3845
XGBoost (seed 7)  baseline R²=-0.1331, +NLP R²=-0.3845


...

CatBoost (seed 0)  baseline R²=-0.1548, +NLP R²=-0.1479
CatBoost (seed 1)  baseline R²=-0.1038, +NLP R²=-0.0215


CatBoost (seed 2)  baseline R²=-0.0560, +NLP R²=-0.1809
CatBoost (seed 3)  baseline R²=-0.1694, +NLP R²=-0.0511


CatBoost (seed 4)  baseline R²=-0.1395, +NLP R²=-0.1100
CatBoost (seed 5)  baseline R²=-0.6079, +NLP R²=-0.3619


CatBoost (seed 6)  baseline R²=-0.2157, +NLP R²=-0.2065
CatBoost (seed 7)  baseline R²=-0.2145, +NLP R²=-0.0995


CatBoost (seed 8)  baseline R²=-0.2322, +NLP R²=-0.1147
CatBoost (seed 9)  baseline R²=-0.0777, +NLP R²=-0.0791


## 4.3 Baseline vs +NLP comparison

In [4]:
reg_df = results.regression_df()
# Mean baseline R² by base model name.
baseline_mean = reg_df[~reg_df["model"].str.endswith("+NLP")].groupby("model")["r2"].mean()
# Mean augmented R² by base model name (remove suffix for alignment).
aug_mean = reg_df[reg_df["model"].str.endswith("+NLP")].assign(
    base_model=lambda d: d["model"].str.replace(" +NLP", "", regex=False)
).groupby("base_model")["r2"].mean()
comparison = pd.concat(
    [baseline_mean.rename("R²_baseline"), aug_mean.rename("R²_+NLP")], axis=1
)
comparison["R²_improvement"] = comparison["R²_+NLP"] - comparison["R²_baseline"]
print("=== Baseline vs NLP-Augmented R² (mean over seeds) ===")
display(comparison.round(6))
reg_df.round(6)

=== Baseline vs NLP-Augmented R² (mean over seeds) ===


,R²_baseline,R²_+NLP,R²_improvement
CatBoost,-0.197143,-0.137334,0.059810
Elastic Net,-0.000309,-0.000309,0.000000
Lasso,-0.000309,-0.000309,0.000000
LightGBM,-0.980772,-1.211791,-0.231019
Linear Regression,-0.013099,-0.008339,0.004760
Ridge,-0.013013,-0.008100,0.004914
XGBoost,-0.133071,-0.384483,-0.251412


,model,seed,mse,rmse,mae,r2,train_time_s,infer_time_s,n_test_samples,wall_train_start_iso,wall_train_end_iso,wall_infer_start_iso,wall_infer_end_iso
0,Linear Regression,0,0.005647,0.075147,0.057257,-0.013099,0.000583,0.000062,622,2026-05-07T22:54:10.014561+00:00,2026-05-07T22:54:10.015151+00:00,2026-05-07T22:54:10.015156+00:00,2026-05-07T22:54:10.015220+00:00
1,Linear Regression +NLP,0,0.005621,0.074970,0.056897,-0.008339,0.000480,0.000046,622,2026-05-07T22:54:10.015670+00:00,2026-05-07T22:54:10.016154+00:00,2026-05-07T22:54:10.016157+00:00,2026-05-07T22:54:10.016204+00:00
2,Linear Regression,1,0.005647,0.075147,0.057257,-0.013099,0.000376,0.000050,622,2026-05-07T22:54:10.016632+00:00,2026-05-07T22:54:10.017012+00:00,2026-05-07T22:54:10.017016+00:00,2026-05-07T22:54:10.017067+00:00
3,Linear Regression +NLP,1,0.005621,0.074970,0.056897,-0.008339,0.000431,0.000044,622,2026-05-07T22:54:10.017449+00:00,2026-05-07T22:54:10.017883+00:00,2026-05-07T22:54:10.017886+00:00,2026-05-07T22:54:10.017931+00:00
4,Linear Regression,2,0.005647,0.075147,0.057257,-0.013099,0.000375,0.000045,622,2026-05-07T22:54:10.018313+00:00,2026-05-07T22:54:10.018691+00:00,2026-05-07T22:54:10.018695+00:00,2026-05-07T22:54:10.018741+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,CatBoost +NLP,7,0.006129,0.078288,0.064255,-0.099546,0.053312,0.000334,622,2026-05-07T22:54:12.635790+00:00,2026-05-07T22:54:12.689111+00:00,2026-05-07T22:54:12.689119+00:00,2026-05-07T22:54:12.689456+00:00
136,CatBoost,8,0.006868,0.082875,0.065577,-0.232183,0.051843,0.000512,622,2026-05-07T22:54:12.690089+00:00,2026-05-07T22:54:12.741942+00:00,2026-05-07T22:54:12.741956+00:00,2026-05-07T22:54:12.742472+00:00
137,CatBoost +NLP,8,0.006214,0.078827,0.064818,-0.114743,0.060491,0.000328,622,2026-05-07T22:54:12.743643+00:00,2026-05-07T22:54:12.804146+00:00,2026-05-07T22:54:12.804154+00:00,2026-05-07T22:54:12.804484+00:00
138,CatBoost,9,0.006007,0.077507,0.061045,-0.077729,0.051526,0.000370,622,2026-05-07T22:54:12.805189+00:00,2026-05-07T22:54:12.856724+00:00,2026-05-07T22:54:12.856733+00:00,2026-05-07T22:54:12.857105+00:00


## 4.4 Persist results

In [5]:
results.save(RESULTS_DIR)
print(f"Total regression experiments: {len(results.regression_results)}")

Results saved to /home/hrilab/energy-analysis-pipeline/results/
Total regression experiments: 140
